In [ ]:
import numpy as np
from typing import Tuple, Dict
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import mean_squared_error,mean_absolute_error
import math
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from EEGNet import EEGNet
from dataset import EEGDataset
from main.backup.cstm import CNN_LSTM

device = torch.device("cuda" if torch.cuda.is_available() else
                          "mps" if torch.backends.mps.is_available() else
                          "cpu")
base_path = '/Users/cirilla/Documents/Code/ml/eeg/files copy'

In [ ]:
def plot_metrics(history):
    epochs = range(1, len(history['train_loss']) + 1)

    plt.figure(figsize=(16, 10))

    # Loss
    plt.subplot(2, 2, 1)
    plt.plot(epochs, history['train_loss'], label='Train Loss')
    plt.plot(epochs, history['val_loss'], label='Val Loss')
    plt.title('Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    # Accuracy
    plt.subplot(2, 2, 2)
    plt.plot(epochs, history['val_acc'], label='Val Accuracy', color='green')
    plt.title('Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    # MAE
    plt.subplot(2, 2, 3)
    plt.plot(epochs, history['val_mae'], label='Val MAE', color='orange')
    plt.title('Validation MAE')
    plt.xlabel('Epoch')
    plt.ylabel('MAE')
    plt.legend()

    # RMSE
    plt.subplot(2, 2, 4)
    plt.plot(epochs, history['val_rmse'], label='Val RMSE', color='red')
    plt.title('Validation RMSE')
    plt.xlabel('Epoch')
    plt.ylabel('RMSE')
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_test_metrics(test_metrics: Dict[str, float]):
    labels = list(test_metrics.keys())
    values = list(test_metrics.values())

    plt.figure(figsize=(8, 5))
    bars = plt.bar(labels, values, color=['skyblue', 'lightgreen', 'orange', 'salmon', 'red'])
    plt.title("Test Set Metrics")
    plt.ylabel("Score")
    plt.ylim(0, max(values) * 1.2)

    # Add value labels on top
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.0, height, f"{height:.3f}", ha='center', va='bottom')

    plt.tight_layout()
    plt.show()

In [ ]:
def train(model, train_loader, optimizer, criterion, device):
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()

        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == y).sum().item()
        total += y.size(0)
    epoch_loss = train_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc